# DSA 210 – Internet Usage and Economic Development
## EDA & Hypothesis Testing
**Student:** Serkan Evleksiz  
**Dataset:** OECD Countries, 2005–2020  
**Sources:** World Bank Open Data, Our World in Data, OECD

---
## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('../figures', exist_ok=True)

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df = pd.read_csv('../data/processed/oecd_internet_gdp.csv')
print(f'Shape: {df.shape}')
df.head(10)

---
## 1. Basic Overview

In [ ]:
print('=== Dataset Info ===')
print(f'Countries : {df["country"].nunique()}')
print(f'Years     : {df["year"].min()} – {df["year"].max()}')
print(f'Total rows: {len(df)}')
print()
print('=== Descriptive Statistics ===')
df[['internet_usage_pct', 'gdp_per_capita_usd']].describe().round(2)

In [ ]:
# Missing value check
print('=== Missing Values ===')
print(df.isnull().sum())

---
## 2. Internet Usage Over Time

In [ ]:
yearly_avg = df.groupby('year')[['internet_usage_pct', 'gdp_per_capita_usd']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(yearly_avg.index, yearly_avg['internet_usage_pct'], 'o-', color='steelblue', linewidth=2)
axes[0].set_title('Average Internet Usage (OECD, 2005–2020)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('% of individuals using internet')
axes[0].set_ylim(0, 100)

axes[1].plot(yearly_avg.index, yearly_avg['gdp_per_capita_usd'] / 1000, 'o-', color='coral', linewidth=2)
axes[1].set_title('Average GDP per Capita (OECD, 2005–2020)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('GDP per Capita (thousand USD)')

plt.tight_layout()
plt.savefig('../figures/01_trends_over_time.png', bbox_inches='tight')
plt.show()
print('Saved: figures/01_trends_over_time.png')

---
## 3. Country-Level Snapshot (2020)

In [ ]:
df_2020 = df[df['year'] == 2020].copy()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    df_2020['internet_usage_pct'],
    df_2020['gdp_per_capita_usd'] / 1000,
    c=df_2020['gdp_per_capita_usd'],
    cmap='viridis',
    s=80,
    alpha=0.85
)

# Label a few notable countries
for _, row in df_2020.iterrows():
    if row['country'] in ['United States', 'Turkey', 'Luxembourg', 'Mexico', 'Norway', 'Germany', 'South Korea']:
        ax.annotate(row['country'], (row['internet_usage_pct'], row['gdp_per_capita_usd']/1000),
                    fontsize=8, xytext=(4, 4), textcoords='offset points')

plt.colorbar(scatter, label='GDP per Capita (USD)')
ax.set_xlabel('Internet Usage (%)')
ax.set_ylabel('GDP per Capita (thousand USD)')
ax.set_title('Internet Usage vs GDP per Capita – OECD Countries (2020)')
plt.tight_layout()
plt.savefig('../figures/02_scatter_2020.png', bbox_inches='tight')
plt.show()
print('Saved: figures/02_scatter_2020.png')

---
## 4. Correlation Analysis

In [ ]:
# Pearson correlation (all years)
r_all, p_all = stats.pearsonr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
print(f'Pearson r (all years): {r_all:.3f}  (p = {p_all:.4f})')

# Per-year correlation
yearly_corr = df.groupby('year').apply(
    lambda g: stats.pearsonr(g['internet_usage_pct'], g['gdp_per_capita_usd'])[0]
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yearly_corr.index, yearly_corr.values, color='steelblue', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Pearson r')
ax.set_title('Yearly Pearson Correlation: Internet Usage vs GDP per Capita')
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.savefig('../figures/03_yearly_correlation.png', bbox_inches='tight')
plt.show()
print('Saved: figures/03_yearly_correlation.png')

In [ ]:
# Correlation matrix heatmap
df_log = df.copy()
df_log['log_gdp'] = np.log(df['gdp_per_capita_usd'])

corr_matrix = df_log[['internet_usage_pct', 'gdp_per_capita_usd', 'log_gdp', 'year']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../figures/04_correlation_matrix.png', bbox_inches='tight')
plt.show()
print('Saved: figures/04_correlation_matrix.png')

---
## 5. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Internet usage distribution
axes[0, 0].hist(df['internet_usage_pct'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_xlabel('Internet Usage (%)')
axes[0, 0].set_title('Distribution of Internet Usage')

# GDP distribution
axes[0, 1].hist(df['gdp_per_capita_usd'] / 1000, bins=25, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].set_xlabel('GDP per Capita (thousand USD)')
axes[0, 1].set_title('Distribution of GDP per Capita')

# Log GDP distribution
axes[1, 0].hist(np.log(df['gdp_per_capita_usd']), bins=25, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[1, 0].set_xlabel('Log(GDP per Capita)')
axes[1, 0].set_title('Distribution of Log GDP per Capita')

# Boxplot by year (internet usage)
years_sample = [2005, 2008, 2011, 2014, 2017, 2020]
data_by_year = [df[df['year'] == y]['internet_usage_pct'].values for y in years_sample]
axes[1, 1].boxplot(data_by_year, labels=years_sample, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Internet Usage (%)')
axes[1, 1].set_title('Internet Usage Distribution by Year')

plt.tight_layout()
plt.savefig('../figures/05_distributions.png', bbox_inches='tight')
plt.show()
print('Saved: figures/05_distributions.png')

---
## 6. Country Trajectories (Selected Countries)

In [ ]:
highlight = ['United States', 'Germany', 'Turkey', 'Mexico', 'South Korea', 'Norway']
df_hi = df[df['country'].isin(highlight)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for country, grp in df_hi.groupby('country'):
    grp = grp.sort_values('year')
    axes[0].plot(grp['year'], grp['internet_usage_pct'], marker='o', markersize=3, label=country)
    axes[1].plot(grp['year'], grp['gdp_per_capita_usd'] / 1000, marker='o', markersize=3, label=country)

axes[0].set_title('Internet Usage Over Time (Selected Countries)')
axes[0].set_ylabel('Internet Usage (%)')
axes[0].legend(fontsize=8)

axes[1].set_title('GDP per Capita Over Time (Selected Countries)')
axes[1].set_ylabel('GDP per Capita (thousand USD)')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.set_xlabel('Year')

plt.tight_layout()
plt.savefig('../figures/06_country_trajectories.png', bbox_inches='tight')
plt.show()
print('Saved: figures/06_country_trajectories.png')

---
## 7. Hypothesis Testing

### H1: There is a significant positive correlation between internet usage and GDP per capita
- **H₀:** ρ = 0 (no correlation)
- **H₁:** ρ > 0 (positive correlation)
- **Test:** Pearson correlation test (parametric), Spearman correlation test (non-parametric)

In [ ]:
# H1: Pearson correlation
r, p = stats.pearsonr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
print('=== H1: Pearson Correlation Test ===')
print(f'r = {r:.4f}')
print(f'p = {p:.6f}')
print(f'Result: {"Reject H₀ → significant positive correlation" if p < 0.05 and r > 0 else "Fail to reject H₀"}')

print()

# Spearman (more robust, non-parametric)
rho, p_sp = stats.spearmanr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
print('=== H1: Spearman Correlation Test ===')
print(f'ρ = {rho:.4f}')
print(f'p = {p_sp:.6f}')
print(f'Result: {"Reject H₀ → significant positive correlation" if p_sp < 0.05 and rho > 0 else "Fail to reject H₀"}')

### H2: Countries with internet usage above 80% have significantly higher GDP than those below 80%
- **H₀:** Mean GDP of high-internet group = Mean GDP of low-internet group
- **H₁:** Mean GDP of high-internet group > Mean GDP of low-internet group
- **Test:** Independent samples t-test

In [ ]:
threshold = 80
high = df[df['internet_usage_pct'] >= threshold]['gdp_per_capita_usd']
low  = df[df['internet_usage_pct'] <  threshold]['gdp_per_capita_usd']

t_stat, p_val = stats.ttest_ind(high, low, alternative='greater')

print('=== H2: Independent Samples t-test ===')
print(f'Threshold: {threshold}% internet usage')
print(f'High group (n={len(high)}): mean GDP = ${high.mean():,.0f}')
print(f'Low  group (n={len(low)}):  mean GDP = ${low.mean():,.0f}')
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.6f}')
print(f'Result: {"Reject H₀ → high-internet countries have significantly higher GDP" if p_val < 0.05 else "Fail to reject H₀"}')

# Visualize
fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot([low/1000, high/1000], labels=[f'Internet < {threshold}%', f'Internet ≥ {threshold}%'],
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('GDP per Capita (thousand USD)')
ax.set_title(f'GDP per Capita by Internet Usage Group (threshold = {threshold}%)')
plt.tight_layout()
plt.savefig('../figures/07_h2_ttest_boxplot.png', bbox_inches='tight')
plt.show()
print('Saved: figures/07_h2_ttest_boxplot.png')

### H3: Internet usage has increased significantly from 2005 to 2020 across OECD countries
- **H₀:** Mean internet usage in 2005 = Mean internet usage in 2020
- **H₁:** Mean internet usage in 2020 > Mean internet usage in 2005
- **Test:** Paired samples t-test (same countries, different years)

In [ ]:
df_2005 = df[df['year'] == 2005].set_index('country_code')['internet_usage_pct']
df_2020 = df[df['year'] == 2020].set_index('country_code')['internet_usage_pct']

common = df_2005.index.intersection(df_2020.index)
x2005 = df_2005.loc[common]
x2020 = df_2020.loc[common]

t_stat, p_val = stats.ttest_rel(x2020, x2005, alternative='greater')

print('=== H3: Paired t-test (2005 vs 2020) ===')
print(f'Mean internet usage 2005: {x2005.mean():.1f}%')
print(f'Mean internet usage 2020: {x2020.mean():.1f}%')
print(f'Mean increase: +{(x2020 - x2005).mean():.1f} percentage points')
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.8f}')
print(f'Result: {"Reject H₀ → internet usage has significantly increased" if p_val < 0.05 else "Fail to reject H₀"}')

---
## 8. Summary of Results

In [ ]:
print('=' * 65)
print('HYPOTHESIS TESTING SUMMARY')
print('=' * 65)
print()
print('H1: Internet usage positively correlates with GDP per capita')
r, p = stats.pearsonr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
print(f'    Pearson r={r:.3f}, p={p:.4f} → {"CONFIRMED" if p < 0.05 else "NOT CONFIRMED"}')
print()
high = df[df['internet_usage_pct'] >= 80]['gdp_per_capita_usd']
low  = df[df['internet_usage_pct'] <  80]['gdp_per_capita_usd']
t_stat, p_val = stats.ttest_ind(high, low, alternative='greater')
print('H2: High-internet countries (≥80%) have higher GDP')
print(f'    t={t_stat:.3f}, p={p_val:.4f} → {"CONFIRMED" if p_val < 0.05 else "NOT CONFIRMED"}')
print()
df_2005 = df[df['year'] == 2005].set_index('country_code')['internet_usage_pct']
df_2020_v = df[df['year'] == 2020].set_index('country_code')['internet_usage_pct']
common = df_2005.index.intersection(df_2020_v.index)
t_stat3, p_val3 = stats.ttest_rel(df_2020_v.loc[common], df_2005.loc[common], alternative='greater')
print('H3: Internet usage increased significantly from 2005 to 2020')
print(f'    t={t_stat3:.3f}, p={p_val3:.6f} → {"CONFIRMED" if p_val3 < 0.05 else "NOT CONFIRMED"}')
print()
print('=' * 65)